### Silver Transormation


In [15]:
today_date = '2025-12-30'

StatementMeta(, 833e71ad-eb7e-49d3-b6f7-d2e2de59a07e, 17, Finished, Available, Finished)

In [16]:
Fabric_bronze_path = 'abfss://Fabric_Dev@onelake.dfs.fabric.microsoft.com/Fabric_LH_Sales.Lakehouse/Tables/dbo/tblsales_bronze'

from pyspark.sql.functions import col

df = spark.read.format('delta').load(Fabric_bronze_path).filter(col('processing_date')==str(today_date))

StatementMeta(, 833e71ad-eb7e-49d3-b6f7-d2e2de59a07e, 18, Finished, Available, Finished)

In [17]:
display(df)

StatementMeta(, 833e71ad-eb7e-49d3-b6f7-d2e2de59a07e, 19, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 9564e14d-5208-4c8c-acee-853e9e6105ac)

### Data Cleaning

##### Handing Duplicates

In [18]:
#print('Before removing Duplicate',df.count())
df_remove_duplicate = df.dropDuplicates()
#print('AFter removing Duplicate',df.count())


StatementMeta(, 833e71ad-eb7e-49d3-b6f7-d2e2de59a07e, 20, Finished, Available, Finished)

### Handle Missing

##### Drop rows with missing critical values

In [19]:
df_dropped =df_remove_duplicate.dropna(subset=['Order_ID','Customer_ID'])

StatementMeta(, 833e71ad-eb7e-49d3-b6f7-d2e2de59a07e, 21, Finished, Available, Finished)

### Business Transformation

##### Delivery Days

In [20]:
df_days = df_dropped.withColumn('Delivery_Days',(col('Ship_Date')-col('Order_Date')).cast('int'))

StatementMeta(, 833e71ad-eb7e-49d3-b6f7-d2e2de59a07e, 22, Finished, Available, Finished)

In [21]:
display(df_days)

StatementMeta(, 833e71ad-eb7e-49d3-b6f7-d2e2de59a07e, 23, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 75f90875-45ca-401f-bd5f-508ecf995aad)

In [22]:
df_ProfitMargin = df_days.withColumn('Profit_Margin',col('Profit')/col('Sales'))

StatementMeta(, 833e71ad-eb7e-49d3-b6f7-d2e2de59a07e, 24, Finished, Available, Finished)

In [23]:
display(df_ProfitMargin)

StatementMeta(, 833e71ad-eb7e-49d3-b6f7-d2e2de59a07e, 25, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 971362d1-5107-40a2-b74b-3d3cfa8b68fe)

### Writting to silver

In [24]:
df_ProfitMargin.createOrReplaceTempView('t_silver_new_data')

StatementMeta(, 833e71ad-eb7e-49d3-b6f7-d2e2de59a07e, 26, Finished, Available, Finished)

In [25]:
%%sql
SELECT * from t_silver_new_data

StatementMeta(, 833e71ad-eb7e-49d3-b6f7-d2e2de59a07e, 27, Finished, Available, Finished)

<Spark SQL result set with 1000 rows and 29 fields>

In [26]:
Fabric_tblsales_silver = 'abfss://Fabric_Dev@onelake.dfs.fabric.microsoft.com/Fabric_LH_Sales.Lakehouse/Tables/dbo/tblsales_silver'
try:
    spark.read.format('delta').load(Fabric_tblsales_silver).createOrReplaceTempView('t_tblsales_silver')
except:
  v_create_table = f"""CREATE TABLE IF NOT EXISTS tblsales_silver (
            Row_ID string,
            Order_ID string,
            Order_Date date,
            Ship_Date date,
            Ship_Mode string,
            Customer_ID string,
            Customer_Name string,
            Segment string,
            Postal_Code string,
            City string,
            State string,
            Country string,
            Region string,
            Market string,
            Product_ID string,
            Category string,
            Sub_Category string,
            Product_Name string,
            Sales DOUBLE,
            Quantity int,
            Discount DOUBLE,
            Profit DOUBLE,
            Shipping_Cost DOUBLE,
            Order_Priority string,
            Month string,
            Year string,
            processing_date date,
            Delivery_Days int,
            Profit_Margin DOUBLE
    
    )"""

spark.sql(v_create_table)
    #spark.read.format('delta').load(fabric_tblsales_silver).createOrReplaceTempView('t_tblsales_silver')


StatementMeta(, 833e71ad-eb7e-49d3-b6f7-d2e2de59a07e, 28, Finished, Available, Finished)

DataFrame[]

In [28]:


sql_statement = f"""MERGE INTO tblsales_silver as target
                    USING t_silver_new_data as source
                    on target.Order_ID = source.Order_ID and target.Customer_ID = source.Customer_ID

                    WHEN MATCHED THEN
                        UPDATE SET 
                        target.Row_ID = source.Row_ID,
                        target.Order_ID = source.Order_ID,
                        target.Order_Date = source.Order_Date,
                        target.Ship_Date = source.Ship_Date,
                        target.Ship_Mode = source.Ship_Mode,
                        target.Customer_ID = source.Customer_ID,
                        target.Customer_Name = source.Customer_Name,
                        target.Segment = source.Segment,
                        target.Postal_Code = source.Postal_Code,
                        target.City = source.City,
                        target.State = source.State,
                        target.Country = source.Country,
                        target.Region = source.Region,
                        target.Market = source.Market,
                        target.Product_ID = source.Product_ID,
                        target.Category = source.Category,
                        target.Sub_Category = source.Sub_Category,
                        target.Product_Name = source.Product_Name,
                        target.Sales = source.Sales,
                        target.Quantity = source.Quantity,
                        target.Discount = source.Discount,
                        target.Profit = source.Profit,
                        target.Shipping_Cost = source.Shipping_Cost,
                        target.Order_Priority = source.Order_Priority,
                        target.Month = source.Month,
                        target.Year = source.Year,
                        target.processing_date = source.processing_date,
                        target.Delivery_Days = source.Delivery_Days,
                        target.Profit_Margin = source.Profit_Margin



                    WHEN NOT MATCHED THEN
                         INSERT (Row_ID,
                                 Order_ID,
                                 Order_Date,
                                 Ship_Date,
                                 Ship_Mode,
                                 Customer_ID,
                                 Customer_Name,
                                 Segment,
                                 Postal_Code,
                                 City,
                                 State,
                                 Country,
                                 Region,
                                 Market,
                                 Product_ID,
                                 Category,
                                 Sub_Category,
                                 Product_Name,
                                 Sales,
                                 Quantity,
                                 Discount,
                                 Profit,
                                 Shipping_Cost,
                                 Order_Priority,
                                 Month,
                                 Year,
                                 processing_date,
                                 Delivery_Days,
                                 Profit_Margin)
                                 VALUES
                                 (
                                 source.Row_ID,
                                 source.Order_ID,
                                 source.Order_Date,
                                 source.Ship_Date,
                                 source.Ship_Mode,
                                 source.Customer_ID,
                                 source.Customer_Name,
                                 source.Segment,
                                 source.Postal_Code,
                                 source.City,
                                 source.State,
                                 source.Country,
                                 source.Region,
                                 source.Market,
                                 source.Product_ID,
                                 source.Category,
                                 source.Sub_Category,
                                 source.Product_Name,
                                 source.Sales,
                                 source.Quantity,
                                 source.Discount,
                                 source.Profit,
                                 source.Shipping_Cost,
                                 source.Order_Priority,
                                 source.Month,
                                 source.Year,
                                 source.processing_date,
                                 source.Delivery_Days,
                                 source.Profit_Margin
                                 )"""
spark.sql(sql_statement).show()


StatementMeta(, 833e71ad-eb7e-49d3-b6f7-d2e2de59a07e, 30, Finished, Available, Finished)

+-----------------+----------------+----------------+-----------------+
|num_affected_rows|num_updated_rows|num_deleted_rows|num_inserted_rows|
+-----------------+----------------+----------------+-----------------+
|             1675|               0|               0|             1675|
+-----------------+----------------+----------------+-----------------+



In [29]:
%%sql
select * from tblsales_silver

StatementMeta(, 833e71ad-eb7e-49d3-b6f7-d2e2de59a07e, 31, Finished, Available, Finished)

<Spark SQL result set with 1000 rows and 29 fields>